# [1교시]

## 1. 언어 모델의 직관적 이해

언어 모델은 문장이 얼마나 자연스러운지 확률을 구하거나, 이전 단어들이 주어졌을 때 다음 단어를 예측하는 시스템입니다. 수식 없이 직관적으로 접근해 봅니다.

스마트폰 키보드에서 **나는**이라는 단어와 **오늘**이라는 단어를 연이어 입력했을 때, 키보드는 다음 단어로 **밥을**또는 **영화를**추천합니다. 

우리의 전체 데이터가 다음과 같다고 가정합니다.
- 데이터 1: 나는 오늘 밥을 먹었다
- 데이터 2: 나는 오늘 영화를 보았다

여기서 **나는**다음 **오늘**이라는 흐름 뒤에 등장한 단어는 **밥을** 1번, **영화를** 1번입니다. 두 단어 모두 등장할 확률이 반반입니다. 이렇게 과거 단어들의 패턴을 세어서 예측하는 방식이 통계적 언어 모델입니다.

## 2. N-gram의 개념

다음 단어를 예측하기 위해 앞에 몇 개의 단어를 참고할 것인지 결정하는 것이 **N-gram**입니다.
- 1-gram (Unigram): 이전 단어를 참고하지 않고 가장 많이 나온 단어를 찍습니다.
- 2-gram (Bigram): 바로 앞 1개의 단어만 참고합니다.
- 3-gram (Trigram): 바로 앞 2개의 단어를 참고합니다.

In [1]:
sentences = [
    '나는 오늘 밥을 먹었다',
    '나는 오늘 영화를 보았다'
]

# 공백으로 분류
words_list = [sentence.split() for sentence in sentences]
words_list

[['나는', '오늘', '밥을', '먹었다'], ['나는', '오늘', '영화를', '보았다']]

In [2]:
next_words = []
for words in words_list:
    for i in range(len(words)-2):
        if words[i] == '나는' and words[i+1] == '오늘':
            next_words.append(words[i+2])
print(f'나는 오늘 다음에 등장한 단어들 : {next_words}')            

나는 오늘 다음에 등장한 단어들 : ['밥을', '영화를']


## 2. 말뭉치(Corpus)의 편향성과 토큰화(Tokenization)의 종속성

통계적 언어 모델은 철저하게 학습된 말뭉치의 세상만을 인식합니다. 
우리가 가진 전체 데이터가 다음과 같다고 가정합니다.
- 데이터 1: 컴퓨터 공학은 매우 흥미롭다
- 데이터 2: 컴퓨터 공학은 취업이 잘된다

이 세상에서 **컴퓨터**다음 **공학은**이라는 흐름이 나왔을 때, 이 모델은 세상에 존재하는 다음 단어가 오직 **매우**혹은 **취업이**두 가지만 존재한다고 확신합니다. 만약 입력으로 **컴퓨터 공학은 어렵다**라는 문장이 들어오면, 이 모델은 확률을 0으로 만들어버립니다. 이를 통해 우리는 양질의 거대한 말뭉치 구축이 통계적 모델링의 핵심임을 알 수 있습니다.

또한, N-gram 모델은 텍스트를 자르는 단위인 토큰화 방식에 절대적으로 의존합니다.
**나는 오늘 밥을 먹었다**문장을 어절(띄어쓰기) 단위로 자르면 다음과 같습니다.
- [나는, 오늘, 밥을, 먹었다] -> 총 4개의 토큰

반면, 형태소 단위로 자르면 다음과 같이 변합니다.
- [나, 는, 오늘, 밥, 을, 먹, 었, 다] -> 총 8개의 토큰

이때 Bigram(바로 앞 1개 단어 참고)을 구축한다고 하면, 어절 단위에서는 **오늘**다음 **밥을**이 연결되지만, 형태소 단위에서는 **오늘**다음 **밥**이 연결됩니다. 한국어의 경우 교착어의 특성상 띄어쓰기 기반 N-gram은 조사의 결합에 따라 완전히 다른 패턴으로 분산되어 성능이 극도로 저하되므로, 형태소 분석이 필수적으로 동반되어야 함을 직관적으로 이해할 수 있습니다.

## 3. N-gram의 한계: 장기 의존성(Long-term Dependency) 문제

앞의 단어를 참고하여 다음을 예측하는 N-gram은 구조적으로 치명적인 맹점을 가집니다. 바로 문맥이 길어질 때 발생하는 장기 의존성 문제입니다.

- 예시 문장: "그 프랑스 출신 요리사는 서울에 온 지 10년이 넘었지만 여전히 아침마다 고향을 그리워하며 **[?]** 로 중얼거린다"

사람은 이 문장을 읽고 주어가 **프랑스 출신 요리사** 라는 아주 먼 과거의 정보를 기억하여 괄호 안에 **프랑스어** 가 들어갈 확률이 높다고 추론합니다.
하지만 일반적인 통계 모델이 사용하는 Trigram(앞의 2개 단어 참고) 구조에서는 오직 **고향을** 과 **그리워하며** 라는 두 단어만 보고 다음 단어를 찍어야 합니다. 앞의 **프랑스** 라는 정보는 모델의 시야에서 완전히 삭제되었기 때문에 제대로 된 예측이 불가능해집니다. 이 한계를 극복하기 위해 N을 무한정 늘리면, 똑같은 패턴을 과거 데이터에서 단 한 번도 찾지 못하는 희소 문제에 직면하게 됩니다.

## 4. 특수 토큰의 도입: 시작(BOS)과 종료(EOS)

단어와 단어의 연결성만 세다 보면, 문장이 어디서 시작되고 어디서 끝나는지 모델은 알 수 없습니다. 이를 위해 데이터 앞뒤에 특수한 가짜 단어를 삽입해야 합니다.
- `<s>` (Begin of Sentence): 문장의 시작을 알림
- `</s>` (End of Sentence): 문장의 끝을 알림

기존: 나는 밥을 먹었다
전처리 후: `<s>` 나는 밥을 먹었다 `</s>`

이렇게 전처리하면, 모델은 `<s>` 다음에 **나는** 이 올 확률을 계산함으로써 문장의 첫 단어로 어떤 단어들이 주로 등장하는지 통계적으로 학습할 수 있게 됩니다. 마찬가지로 **먹었다** 다음에 `</s>` 가 올 패턴을 통해 문장을 안전하게 끝마치는 법을 터득합니다.

## 1. 말뭉치 특수 토큰 부착 및 기본 토큰화
문장의 시작을 의미하는 `<s>` 와 끝을 의미하는 `</s>` 를 부착하여 모델이 문장의 경계를 인식할 수 있도록 데이터를 정제합니다.

In [3]:
raw_corpus = [
    '나는 오늘 파이토치 학습을 한다',
    '나는 내일 자연어 처리를 한다',
    '오늘 파이토치 학습은 매우 즐겁다',
    '파이토치 코딩은 매우 재미있다'
]
# BOS(Begin of Sequence) EOS(End of Sequence) 부착 함수
def process_corpus(corpus):
    processed = []
    for sentence in corpus:
        processed.append(['<s>']  + sentence.split() + ['</s>'] )
    return processed

tokenized_corpus = process_corpus(raw_corpus)
tokenized_corpus

[['<s>', '나는', '오늘', '파이토치', '학습을', '한다', '</s>'],
 ['<s>', '나는', '내일', '자연어', '처리를', '한다', '</s>'],
 ['<s>', '오늘', '파이토치', '학습은', '매우', '즐겁다', '</s>'],
 ['<s>', '파이토치', '코딩은', '매우', '재미있다', '</s>']]

## 2. N-gram 추출기 구현 및 데이터베이스화
Python의 `collections` 모듈과 제너레이터 방식을 활용하여, N의 크기에 상관없이 유연하게 N-gram 조각들을 추출하고 빈도를 카운트하는 범용 알고리즘을 설계합니다.

In [4]:
from collections import Counter
def extract_ngrams(tokenized_corpus, n):
    ngram_counts = Counter()
    for tokens in tokenized_corpus:
        # 토큰길이가 N보다 작으면 추출 불가
        if len(tokens) < n:
            continue
        # 슬라이딩 윈도우 방식
        for i in range(len(tokens) - n+1):
            ngram = tuple(tokens[i : i + n])
            ngram_counts[ngram] += 1
    return ngram_counts

In [5]:
# unigram(1-gram) 추출
unigram = extract_ngrams(tokenized_corpus, 1)
bigram = extract_ngrams(tokenized_corpus, 2)

In [6]:
unigram.most_common(5), bigram.most_common(5)

([(('<s>',), 4), (('</s>',), 4), (('파이토치',), 3), (('나는',), 2), (('오늘',), 2)],
 [(('<s>', '나는'), 2),
  (('오늘', '파이토치'), 2),
  (('한다', '</s>'), 2),
  (('나는', '오늘'), 1),
  (('파이토치', '학습을'), 1)])

# [2교시]

## 3. 문맥(Context) 기반 다음 단어 예측 시뮬레이션
추출된 N-gram 데이터베이스를 활용하여, 특정한 앞 단어(Context)가 주어졌을 때 뒤이어 올 단어들의 분포를 확인하여 통계적 언어 모델의 원형을 모사합니다.

In [7]:
def simulate_prediction(ngram_counts, context_words):
    context_len = len(context_words)
    found = False
    # 튜플의 앞부분이 context_words와 일치하는 n-gram을 탐색
    for ngram, count in ngram_counts.items():
        if list(ngram[:context_len]) == context_words:
            next_word = ngram[context_len]
            print(f'--> 예측단어 : {next_word} 등장횟수 : {count}번')
            found = True
    if not found:
        print(f' --> 데이터베이스에 해당 문맥이 존재하지 않습니다(희소 문제 발생)')

In [8]:
# 나는 다음에 올 단어 예측(Bigram, 2-gram)
simulate_prediction(bigram, ['나는'])

--> 예측단어 : 오늘 등장횟수 : 1번
--> 예측단어 : 내일 등장횟수 : 1번


In [9]:
# Trigram
trigram = extract_ngrams(tokenized_corpus, 3)
simulate_prediction(trigram, ['나는', '오늘'])

--> 예측단어 : 파이토치 등장횟수 : 1번


In [10]:
# 희소문제 
simulate_prediction(trigram, ['파이토치', '처리를'])

 --> 데이터베이스에 해당 문맥이 존재하지 않습니다(희소 문제 발생)


In [11]:
four_gram = trigram = extract_ngrams(tokenized_corpus, 4)
simulate_prediction(four_gram, ['나는', '오늘', '파이토치'])

--> 예측단어 : 학습을 등장횟수 : 1번


## 1. 조건부 확률

우리가 일상에서 친구와 대화할 때를 떠올려 봅시다. 친구가 "나는 오늘 점심으로 맛있는..." 이라고 말하다가 멈췄다면, 우리는 자연스럽게 뒤에 **피자를** 혹은 **햄버거를** 이라는 단어가 올 것이라고 예상합니다. 왜 **책상을** 이라는 단어는 전혀 예상하지 않을까요?

우리의 뇌는 과거 수십 년 동안 들었던 수많은 한국어 문장 데이터를 바탕으로, "맛있는" 이라는 특정 조건이 주어졌을 때 뒤에 "음식"이 등장한 횟수를 무의식적으로 계산하고 있기 때문입니다. 

만약 우리가 100권의 소설책 데이터를 컴퓨터로 분석했다고 가정해 봅시다. 
소설책 전체에서 **맛있는** 이라는 단어가 총 10번 등장했습니다. 
그리고 그 10번 중에서 **맛있는** 바로 뒤에 **피자를** 이 등장한 횟수가 7번이었습니다.
그렇다면 우리는 아주 단순한 나눗셈을 통해, **맛있는** 다음 **피자를** 이 올 확률이 10번 중 7번, 즉 70%라고 결론을 내릴 수 있습니다. 이것이 통계적 언어 모델이 확률을 계산하는 가장 원초적이고 직관적인 방식입니다.

## 1. 확률의 연쇄 법칙과 언어 모델의 본질

앞선 직관적 계산법을 바탕으로 이제 정식 알고리즘 수식을 도입해 보겠습니다.
언어 모델이 문장 전체의 타당성을 평가하는 수학적 원리는 결합 확률(Joint Probability)에 근거합니다. 단어 시퀀스 $W = w_1, w_2, ..., w_n$ 이 주어졌을 때, 이 문장이 나타날 전체 확률 $P(W)$는 연쇄 법칙(Chain Rule)에 의해 다음과 같이 분해됩니다.

$P(w_1, w_2, ..., w_n) = P(w_1) \times P(w_2 | w_1) \times \dots \times P(w_n | w_1, ..., w_{n-1})$

하지만 이 수식은 완벽함에도 불구하고 치명적인 연산의 한계를 지닙니다. 문장이 길어질수록 조건부에 해당하는 과거 단어의 나열이 무한정 길어지며, 학습 말뭉치에서 이와 완벽하게 동일한 긴 패턴을 찾을 확률은 사실상 0에 수렴하기 때문입니다.

## 2. 마르코프 가정(Markov Assumption)의 도입

이러한 연산의 마비를 해결하기 위해 러시아 수학자 마르코프의 이론을 차용합니다. 직전의 단어 몇 개만 잘라서 확률을 계산하겠다는 실무적 타협안입니다. 이를 N-gram에 적용하면 수식이 비약적으로 단순화됩니다.

- 1-gram (Unigram): 이전 문맥을 완전히 무시합니다. $P(w_n)$
- 2-gram (Bigram): 직전 1개 단어만 의존합니다. $P(w_n | w_{n-1})$
- 3-gram (Trigram): 직전 2개 단어만 의존합니다. $P(w_n | w_{n-2}, w_{n-1})$

## 3. 최대 우도 추정(MLE) 기반의 확률 계산과 증명

통계적 모델에서 확률을 구하는 정석은 말뭉치에 등장한 빈도(Count)를 활용한 최대 우도 추정(MLE)입니다. Bigram 연산식은 다음과 같습니다.

$P(w_n | w_{n-1}) = \frac{Count(w_{n-1}, w_n)}{Count(w_{n-1})}$

**실제 데이터 수학적 증명**
- 말뭉치 1: `<s>` 나는 오늘 파이토치 학습을 한다 `</s>`
- 말뭉치 2: `<s>` 나는 내일 자연어 처리를 한다 `</s>`

위 말뭉치에서 **나는** 다음에 **오늘** 이 올 확률을 MLE 방식으로 계산합니다.
분모 $Count(나는)$을 탐색합니다. 말뭉치 1과 2에서 총 2회 등장했습니다.
분자 $Count(나는, 오늘)$을 탐색합니다. 말뭉치 1에서만 1회 등장했습니다.
결과적으로 $P(오늘 | 나는) = \frac{1}{2} = 0.5$입니다. 50%의 수학적 확률로 다음 단어를 타당하게 예측해 내며 알고리즘 연산 원리가 완벽히 증명됩니다.

## 1. N-gram 확률 연산 엔진 설계
단순 카운팅을 넘어, 말뭉치를 동적으로 입력받아 분자와 분모 데이터베이스를 자동 구축하는 객체 지향 클래스를 설계합니다.

In [12]:
from collections import Counter
class MLELanguageModel:
    def __init__(self, n):
        self.n = n
        self.ngram_counts = Counter()
        self.context_counts = Counter()
    def train(self, corpus):
        for sentence in corpus:
            tokens = ['<s>'] + sentence.split() +['</s>']
            for i in range(len(tokens) - self.n+1):
                ngram = tuple(tokens[i : i+self.n])
                context = tuple(tokens[i : i+self.n-1])
                self.ngram_counts[ngram] += 1
                self.context_counts[context] +=1
    def get_probability(self, context, target):
        context_tupe = tuple(context)
        ngram_tuple = tuple(context) + (target,)

        numerator = self.ngram_counts.get(ngram_tuple,0)
        denoinator = self.context_counts.get(context_tupe,0)
        if denoinator == 0:
            return 0.0
        return numerator / denoinator

In [13]:
train_corpus = [
    '나는 오늘 파이토치 학습을한다','나는 내일 자연어 처리를 한다','나는 오늘 머신러닝 논문을 읽는다'
]
bigram_model = MLELanguageModel(n=2)
bigram_model.train(train_corpus)

In [14]:
bigram_model.ngram_counts, bigram_model.context_counts

(Counter({('<s>', '나는'): 3,
          ('나는', '오늘'): 2,
          ('오늘', '파이토치'): 1,
          ('파이토치', '학습을한다'): 1,
          ('학습을한다', '</s>'): 1,
          ('나는', '내일'): 1,
          ('내일', '자연어'): 1,
          ('자연어', '처리를'): 1,
          ('처리를', '한다'): 1,
          ('한다', '</s>'): 1,
          ('오늘', '머신러닝'): 1,
          ('머신러닝', '논문을'): 1,
          ('논문을', '읽는다'): 1,
          ('읽는다', '</s>'): 1}),
 Counter({('<s>',): 3,
          ('나는',): 3,
          ('오늘',): 2,
          ('파이토치',): 1,
          ('학습을한다',): 1,
          ('내일',): 1,
          ('자연어',): 1,
          ('처리를',): 1,
          ('한다',): 1,
          ('머신러닝',): 1,
          ('논문을',): 1,
          ('읽는다',): 1}))

In [15]:
# 나는 오늘  논리적 확률 2/3
bigram_model.get_probability(['나는'], '오늘'), 2/3

(0.6666666666666666, 0.6666666666666666)

# [3교시]

# 언어 모델 평가의 첫걸음: 펄플렉서티(PPL)

언어 모델이 문장을 얼마나 '잘' 생성하는지 평가할 때 가장 먼저 마주하게 되는 지표가 바로 **펄플렉서티(Perplexity, PPL)** 입니다. 복잡한 수식 이전에, 우리 뇌가 언어를 이해하는 방식에 빗대어 PPL을  파악해 보겠습니다.

## 1. "얼마나 당황스러운가?" - 헷갈림의 정도

'Perplexity'라는 단어의 사전적 의미는 **'당혹감'** 또는 **'복잡함'** 입니다. 언어 모델 입장에서 생각하면, 다음에 올 단어를 예측할 때 "얼마나 갈팡질팡하고 있는가"를 수치로 나타낸 것입니다.

- **낮은 PPL**: 모델이 다음에 올 단어를 명확하게 예측하고 있음 (확신이 높음)
- **높은 PPL**: 모델이 어떤 단어가 올지 몰라 당황하고 있음 (불확실성이 높음)

## 2. 분기 계수(Branching Factor): 선택지의 개수

PPL 수치를 가장 쉽게 이해하는 방법은 **'선택지의 평균 개수'** 로 생각하는 것입니다.

예를 들어, 어떤 모델의 PPL이 **10** 이 나왔다면 이는 다음과 같은 의미입니다.
> "이 모델은 매 단계마다 평균적으로 **10개의 선택지** 중에서 어떤 것이 정답인지 고민하고 있다

### 객관식 시험 비유
- **PPL = 2**: 마치 2지선다형(O/X) 문제를 푸는 것과 같습니다. 정답을 맞히기 매우 쉽고 명확합니다.
- **PPL = 100**: 마치 100지선다형 문제를 푸는 것과 같습니다. 정답을 맞히기가 훨씬 어렵고 모델은 매우 혼란스러운 상태입니다.

따라서 좋은 언어 모델일수록 문맥을 날카롭게 파악하여 이 '선택지의 개수(분기 계수)'를 1에 가깝게 줄여나갑니다.

## 3. 문맥(Context)의 힘

똑같은 단어라도 문맥에 따라 PPL은 달라집니다.

*   **문맥이 없는 경우**: "나는 [ ? ]"
    *   '밥을', '학교에', '너를', '어제' 등 수많은 단어가 올 수 있어 PPL이 매우 높습니다.
*   **문맥이 풍부한 경우**: "배가 너무 고파서 식당에 가자마자 [ ? ]"
    *   '주문을', '밥을' 등 몇 가지 단어로 압축됩니다. PPL이 급격히 낮아집니다.

숙련된 모델일수록 앞선 단어들을 통해 후보군을 좁히는 능력이 탁월하며, 결과적으로 전체 문장에 대한 PPL을 낮게 유지합니다.

## 요약
- PPL은 모델의 **불확실성** 을 측정하는 지표입니다.
- **분기 계수** 관점에서 보면, 모델이 고민하는 평균 선택지 개수를 의미합니다.
- **수치가 낮을수록** 모델의 예측 성능이 뛰어나고 문맥 파악 능력이 좋다는 것을 뜻합니다.

# 펄플렉서티(PPL)의 수학적 기초와 로그 확률의 필요성

펄플렉서티(Perplexity)는 언어 모델의 성능을 확률론적 관점에서 정밀하게 측정하는 지표입니다. 이 문서에서는 PPL의 수식적 정의와 실제 계산 과정에서 발생하는 수치적 문제, 그리고 이를 해결하기 위한 로그 연산의 도입 과정을 상세히 다룹니다.

## 1. PPL의 수학적 정의

언어 모델은 문장 $W = (w_1, w_2, ..., w_N)$이 발생할 확률 $P(W)$를 계산합니다. PPL은 이 문장 확률의 역수에 대해 문장 길이 $N$만큼 기하평균을 취한 값으로 정의됩니다.

$$PPL(W) = P(w_1, w_2, ..., w_N)^{-\frac{1}{N}} = \sqrt[N]{\frac{1}{P(w_1, w_2, ..., w_N)}}$$

체인 룰(Chain Rule)을 적용하여 문장의 결합 확률을 조건부 확률의 곱으로 나타내면 다음과 같습니다.

$$PPL(W) = \sqrt[N]{\frac{1}{\prod_{i=1}^{N} P(w_i | w_1, ..., w_{i-1})}}$$

## 2. 컴퓨터 공학적 한계: 언더플로우(Underflow)

수식 그대로 PPL을 계산하려면 0과 1 사이의 작은 확률값들을 수십, 수백 번 곱해야 합니다.

*   예: 단어 하나의 확률이 0.1일 때, 100개 단어의 문장 확률은 $0.1^{100} = 10^{-100}$이 됩니다.
*   **문제점**: 현대 컴퓨터의 부동소수점(Floating Point) 표현 한계를 넘어서면 값이 **0으로 증발(Underflow)** 해버립니다. 0으로 나누기는 불가능하므로 PPL 계산 자체가 실패하게 됩니다.

## 3. 로그(Log) 함수의 도입과 수식 변형

이 문제를 해결하기 위해 **로그(Log)** 를 사용합니다. 로그의 성질 ($\log(a \times b) = \log a + \log b$)을 이용하면 확률의 곱셈을 로그 확률의 덧셈으로 변환할 수 있습니다.

PPL 수식에 자연로그($\ln$)를 취하고 지수함수($\exp$)로 되돌리는 과정을 거치면 다음과 같은 '안전한' 수식이 완성됩니다.

$$PPL(W) = \exp \left( -\frac{1}{N} \sum_{i=1}^{N} \log P(w_i | w_{1...i-1}) \right)$$

이 수식은 로그 공간에서 덧셈을 수행하므로 언더플로우 위협에서 자유롭습니다.

## 4. 수치적 증명 (Numerical Proof)

모델이 2개의 단어로 이루어진 문장($N=2$)에서 각각의 발생 확률을 **0.1**로 예측했다고 가정해 봅시다.

### A. 일반 수식 계산
1.  문장 확률: $P(W) = 0.1 \times 0.1 = 0.01$
2.  PPL 계산: $PPL = (1 / 0.01)^{1/2} = 100^{0.5} = \mathbf{10}$

### B. 로그 수식 계산 (자연로그 사용)
1.  각 단어의 로그 확률: $\log(0.1) \approx -2.3026$
2.  로그 확률의 합: $-2.3026 + (-2.3026) = -4.6052$
3.  평균 및 부호 반전: $-\frac{1}{2} \times (-4.6052) = 2.3026$
4.  지수 함수 환원: $\exp(2.3026) \approx \mathbf{10}$

**결과**: 두 방식의 결과가 일치함을 알 수 있습니다.

In [16]:
# Unigram
# Bigram
# Trigrma
# Laplace Smooting : 확률이 0되는문제 해결

In [17]:
from konlpy.tag import Okt
from nltk.util import ngrams
from collections import Counter, defaultdict
import pandas as pd

In [18]:
# url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
# df = pd.read_csv(url)
# df.head()

In [19]:
corpus = [
    "나는 오늘 맛있는 점심을 먹었다",
    "오늘 점심을 먹고 카페에 갔다",
    "나는 카페에 가서 맛있는 커피를 마셨다",
    "맛있는 점심을 먹는 것은 행복하다",
    "오늘 날씨는 정말 좋다",
    "나는 오늘 공부를 열심히 했다"
]

$P(w_n | w_{n-1}) = \frac{Count(w_{n-1}, w_n)}{Count(w_{n-1})}$

In [20]:
# Ngram모델  n값에 따라서 unigram bigram trigram모델 생성 통합함수
# 문장의 시작과 끝에 <s> </s>  확률 계산 MLE방식

In [21]:
def build_ngram_model(corpus, n):
    """
    n-gram 모델을 구축하여 조건부 확률을 계산합니다.
    """
    model = defaultdict(lambda: defaultdict(lambda: 0))
    
    for sentence in corpus:
        tokens = sentence.split()
        
        # 문장의 시작과 끝을 알리는 패딩 추가 (<s>: 시작, </s>: 끝)
        ngram_list = list(ngrams(tokens, n, pad_left=True, pad_right=True, 
                                 left_pad_symbol='<s>', right_pad_symbol='</s>'))
        
        for ngram in ngram_list:
            context = ngram[:-1]  # n-1개의 이전 단어들
            target = ngram[-1]   # 현재 단어
            model[context][target] += 1
            
    # 확률 계산 (MLE)
    for context in model:
        total_count = float(sum(model[context].values()))
        for target in model[context]:
            model[context][target] /= total_count
            
    return model

# 모델 생성
bigram_model = build_ngram_model(corpus, 2)
trigram_model = build_ngram_model(corpus, 3)

In [22]:
def check_probability(model, context, target):
    prob = model[tuple(context)].get(target, 0)
    print(f"P('{target}' | {context}) = {prob:.4f}")

print("--- Bigram Test ---")
check_probability(bigram_model, ['오늘'], '점심을')
check_probability(bigram_model, ['나는'], '오늘')

print("\n--- Trigram Test ---")
check_probability(trigram_model, ['맛있는', '점심을'], '먹었다')
check_probability(trigram_model, ['나는', '오늘'], '공부를')

--- Bigram Test ---
P('점심을' | ['오늘']) = 0.2500
P('오늘' | ['나는']) = 0.6667

--- Trigram Test ---
P('먹었다' | ['맛있는', '점심을']) = 0.5000
P('공부를' | ['나는', '오늘']) = 0.5000


In [23]:
def generate_next_word(model, context):
    context = tuple(context)
    if context not in model:
        return "<Unknown>"
    
    # 가장 높은 확률을 가진 단어 추출
    next_word = max(model[context].items(), key=lambda x: x[1])[0]
    return next_word

print(f"'카페에' 다음에 올 단어: {generate_next_word(bigram_model, ['카페에'])}")
print(f"'맛있는 점심을' 다음에 올 단어: {generate_next_word(trigram_model, ['맛있는', '점심을'])}")

'카페에' 다음에 올 단어: 갔다
'맛있는 점심을' 다음에 올 단어: 먹었다


# [4교시]

In [24]:
# Laplace Smooting  희소단어가 나올때 분모가 0되는 현상 방지

def build_ngram_model_with_smoothing(corpus, n):
    model = defaultdict(lambda: defaultdict(lambda: 0))
    vocab = set()
    
    for sentence in corpus:
        tokens = sentence.split()
        vocab.update(tokens)
        ngram_list = list(ngrams(tokens, n, pad_left=True, pad_right=True, 
                                 left_pad_symbol='<s>', right_pad_symbol='</s>'))
        for ngram in ngram_list:
            model[ngram[:-1]][ngram[-1]] += 1
    
    V = len(vocab) + 1 # </s> 기호 포함 분모가 0이 되는 것을 방지하기 위해서 분모에 추가
    
    # Laplace Smoothing 적용 확률 계산
    smoothed_model = defaultdict(lambda: defaultdict(lambda: 0))
    for context in model:
        total_count = sum(model[context].values())
        for target in model[context]:
            # (count + 1) / (total_count + V)
            smoothed_model[context][target] = (model[context][target] + 1) / (total_count + V)
            
    return smoothed_model

smoothed_bigram = build_ngram_model_with_smoothing(corpus, 2)
print("Smoothing 적용 후 '오늘' -> '커피를' (데이터에 없는 조합) 확률:")
print(f"{smoothed_bigram[('오늘',)].get('커피를', 1 / (sum(bigram_model[('오늘',)].values()) + 15)):.4f}")

Smoothing 적용 후 '오늘' -> '커피를' (데이터에 없는 조합) 확률:
0.0625


In [25]:
# 클래스별 모델 학습 : 긍정(Positive)리뷰들로만 구성된 N-gram 모델과 부정리뷰들로만 구성된 N-gram모델을 각각 만든다
# 확률 계산 : 새로운 리뷰가 들어오면 리뷰가 긍정모델에서 확률과 부정모델에서 확률 각각 계산
# 두개 비교해서 높은 값을 가지는 클래스로 분류
# 여러단어들의 화률을 계속 곱하면 값이 0에 한없이 가까워지는 uderflow현상 발생 --> 확률에 log를 취해서 더해준다

In [32]:
import math
def build_ngram_model_with_smoothing(corpus, n):
    model = defaultdict(lambda: defaultdict(lambda: 0))
    vocab = set()
    
    for sentence in corpus:
        tokens = sentence.split()
        vocab.update(tokens)
        ngram_list = list(ngrams(tokens, n, pad_left=True, pad_right=True, 
                                 left_pad_symbol='<s>', right_pad_symbol='</s>'))
        for ngram in ngram_list:
            model[ngram[:-1]][ngram[-1]] += 1
            
    V = len(vocab) + 1 # </s> 기호 포함
    
    smoothed_model = defaultdict(lambda: defaultdict(lambda: 0))
    for context in model:
        total_count = sum(model[context].values())
        for target in model[context]:
            # (count + 1) / (total_count + V)
            smoothed_model[context][target] = (model[context][target] + 1) / (total_count + V)
            
    return smoothed_model, model, V


# 2. 문장의 로그 확률(Log Probability)을 계산하는 함수
def calculate_sentence_log_prob(sentence, smoothed_model, raw_model, V, n):
    tokens = sentence.split()
    ngram_list = list(ngrams(tokens, n, pad_left=True, pad_right=True, 
                             left_pad_symbol='<s>', right_pad_symbol='</s>'))
    
    log_prob = 0.0
    for ngram in ngram_list:
        context = ngram[:-1]
        target = ngram[-1]
        
        # 1. smoothed_model에 확률이 이미 계산되어 있는 경우
        if target in smoothed_model[context]:
            prob = smoothed_model[context][target]
        # 2. 컨텍스트는 아는데 해당 타겟 단어를 처음 보는 경우 (스무딩 적용)
        elif context in raw_model:
            total_count = sum(raw_model[context].values())
            prob = 1 / (total_count + V)
        # 3. 컨텍스트 자체를 처음 보는 경우 (스무딩 적용)
        else:
            prob = 1 / V
            
        log_prob += math.log(prob)
        
    return log_prob


# 3. 감성 예측 함수
def predict_sentiment(sentence, models_info, n=2):
    best_class = None
    max_log_prob = -float('inf')
    
    # 각 클래스(긍정, 부정 등)별로 확률을 계산하여 가장 높은 확률을 가진 클래스 선택
    for label, (smoothed_model, raw_model, V) in models_info.items():
        log_prob = calculate_sentence_log_prob(sentence, smoothed_model, raw_model, V, n)
        
        if log_prob > max_log_prob:
            max_log_prob = log_prob
            best_class = label
            
    return best_class

In [34]:
data = {
    'reviews': [
        '이 커피 진짜 맛있다', '오늘 커피 향이 너무 좋아', '최고의 커피 추천합니다', # 긍정 (1)
        '커피가 너무 쓰다', '오늘 커피 맛이 별로네', '최악의 커피 절대 안 먹어'    # 부정 (0)
    ],
    'target': [1, 1, 1, 0, 0, 0]
}
df = pd.DataFrame(data)

N = 2 # Bigram 사용

# 클래스별로 코퍼스 분리 및 모델 학습
models_info = {}
for label in df['target'].unique():
    # 해당 클래스의 리뷰만 추출하여 코퍼스 생성
    corpus = df[df['target'] == label]['reviews'].tolist()
    # 모델 학습
    models_info[label] = build_ngram_model_with_smoothing(corpus, N)

print("모델 학습 완료!\n")

# 새로운 리뷰 테스트
test_reviews = [
    '오늘 커피 진짜 최고의 맛',
    '이 영화 좋아'
]

for review in test_reviews:
    prediction = predict_sentiment(review, models_info, n=N)
    sentiment = "긍정" if prediction == 1 else "부정"
    print(f"리뷰: '{review}' -> 예측된 감성: {sentiment} (Class: {prediction})")

모델 학습 완료!

리뷰: '오늘 커피 진짜 최고의 맛' -> 예측된 감성: 긍정 (Class: 1)
리뷰: '이 영화 좋아' -> 예측된 감성: 긍정 (Class: 1)


# [5교시]

In [59]:
# 다중분류... 뉴스 카테고리
# DATA_URL = "https://raw.githubusercontent.com/kocohub/korean-hate-speech/master/labeled/train.tsv"

In [60]:
import urllib
url = "https://raw.githubusercontent.com/kocohub/korean-hate-speech/master/labeled/train.tsv"
def download_data():
    urllib.request.urlretrieve(url, 'train_hate_tsv')

download_data()

In [61]:
import pandas as pd
df = pd.read_csv('train_hate_tsv', sep='\t').dropna()
# None 없음 hate 싫음 offensive 공격적
df.head()

,comments,contain_gender_bias,bias,hate
0,(현재 호텔주인 심정) 아18 난 마른하늘에 날벼락맞고 호텔망하게생겼는데 누군 계속...,False,others,hate
1,....한국적인 미인의 대표적인 분...너무나 곱고아름다운모습...그모습뒤의 슬픔을...,False,none,none
2,"...못된 넘들...남의 고통을 즐겼던 넘들..이젠 마땅한 처벌을 받아야지..,그래...",False,none,hate
3,"1,2화 어설펐는데 3,4화 지나서부터는 갈수록 너무 재밌던데",False,none,none
4,1. 사람 얼굴 손톱으로 긁은것은 인격살해이고2. 동영상이 몰카냐? 메걸리안들 생각...,True,gender,hate


In [62]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df['labels'] = label_encoder.fit_transform(df['hate'])
label_encoder.classes_

array(['hate', 'none', 'offensive'], dtype=object)

In [63]:
#학습/테스트데이터 분리(8:2)
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df,test_size=0.2,random_state=42)
train_df.shape, test_df.shape, df.shape

((6316, 5), (1580, 5), (7896, 5))

In [64]:
# 전처리 및 어휘사전(vocab) 구축
from konlpy.tag import Okt
okt = Okt()
# okt.pos('아버지가 방에 들어가신다'), okt.morphs('아버지가 방에 들어가신다'), okt.phrases('아버지가 방에 들어가신다')
def konlpy_tokenizer(text):
    return okt.morphs(text)

train_tokenized = []
for text in train_df['comments']:
    tokens = konlpy_tokenizer(text)
    train_tokenized.append(tokens)

test_tokenized = [ konlpy_tokenizer(text) for text in test_df['comments']]

In [66]:
from collections import Counter
#빈도수 기준 단어사전 생성
vocab_size = 8000
all_tokens = [ token for token_list in train_tokenized  for token in token_list  ]
word_counts = Counter(all_tokens)
vocab = {'<PAD>':0, '<UNK>':1}

for word in set([word for word,cnt in word_counts.most_common(vocab_size-2)]):
    vocab[word] = len(vocab)

# 문장의 길이를 통일  시퀀스
import torch
sequence_len = 30
def text_to_sequence(tokens, max_len=30):
    seq = [vocab.get(token,1) for token in tokens]        
    if len(seq) < max_len:
        seq += [0]*(max_len - len(seq))
    else:
        seq = seq[ : max_len]
    return seq
# tensor 변환
x_train = torch.LongTensor([text_to_sequence(tokens) for tokens in train_tokenized])
y_train = torch.FloatTensor(train_df['labels'].values)

x_test = torch.LongTensor([text_to_sequence(tokens) for tokens in test_tokenized])
y_test = torch.LongTensor(test_df['labels'].values)

# 데이터로더
batch_size = 64
from torch.utils.data import TensorDataset, DataLoader
train_loader = DataLoader( TensorDataset(x_train,y_train), batch_size=batch_size,shuffle=True )
test_loader = DataLoader( TensorDataset(x_test, y_test),batch_size=batch_size )

# [6교시]

In [77]:
# 모델정의
import torch.nn as nn
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class TextClassifier(nn.Module):
    def __init__(self, model_type, vocab_size, embed_dim, hidden_dim, output_dim):
        super(TextClassifier, self).__init__()
        self.model_type = model_type
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        if model_type == 'RNN':
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        elif model_type == 'LSTM':
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        elif model_type == 'GRU':
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        elif model_type == 'CNN':
            self.conv = nn.Conv1d(embed_dim, hidden_dim, kernel_size=3)
            # 이미지 batch, channel, height, width
            # conv1d  batch, channel, length
            # 텍스트임베딩은 단어순서가 보통 2번째에 위치--> 채널위치하고 바꿔주는 작업
            # kernel_size=3 은 3-grame

            
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x):
        embedded = self.embedding(x)  # [64, 30, 128]  [Batch, seq_len ,embeded_dim]    
        if self.model_type =='CNN':
            embedded = embedded.transpose(1,2)
            conveled = torch.relu(self.conv(embedded))
            hidden_out = torch.max(conveled, dim=2)[0]
        elif self.model_type == 'LSTM':
            _,(hidden,_) = self.rnn(embedded)
            hidden_out = hidden[-1]        
        else: # RNN OR GRU
            output,hidden = self.rnn(embedded) # torch.Size([64, 30, 256]) torch.Size([1, 64, 256])       
            hidden_out = hidden[-1]
        return self.fc(hidden_out)  # 64,256


In [78]:
# model_type = 'LSTM'
# embed_dim = 128
# hidden_dim = 256
# output_dim = len(y_train.unique())
# tc = TextClassifier(model_type, vocab_size, embed_dim, hidden_dim, output_dim).to(device)
# tc.train()
# for b_x, b_y in train_loader:
#     b_x, b_y = b_x.to(device), b_y.to(device)
#     output = tc(b_x)
#     print(output.shape)
#     break

# [7교시]

In [81]:
# 학습 및 평가용
def train_and_evaluation(model_type ='RNN',epochs=2):
    embed_dim = 128
    hidden_dim = 256
    output_dim = len(y_train.unique())
    tc = TextClassifier(model_type, vocab_size, embed_dim, hidden_dim, output_dim).to(device)
    criterion = nn.CrossEntropyLoss() # softmax 내장
    optimizer = torch.optim.Adam(tc.parameters())
    for epoch in range(epochs):
        tc.train()
        total_loss = 0
        for b_x, b_y in train_loader:
            optimizer.zero_grad()
            b_x, b_y = b_x.to(device), b_y.long().to(device)
            output = tc(b_x)  # forward
            loss = criterion(output,b_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        # 검증
        tc.eval()
        correct = 0
        with torch.no_grad():
            for b_x, b_y in test_loader:
                b_x, b_y = b_x.to(device), b_y.to(device)
                outputs = tc(b_x)
                _,pred = torch.max(outputs,1)
                correct += (pred == b_y).sum().item()
        acc = correct / len(test_df)
        print(f'epoch {epoch+1} / {epochs}  loss : {total_loss / len(train_loader):.4f}  acc : {acc:.4f}')
        


In [82]:
train_and_evaluation('RNN',10)

epoch 1 / 10  loss : 1.0747  acc : 0.4380
epoch 2 / 10  loss : 1.0369  acc : 0.4468
epoch 3 / 10  loss : 1.0224  acc : 0.3899
epoch 4 / 10  loss : 0.9879  acc : 0.4646
epoch 5 / 10  loss : 0.9726  acc : 0.4671
epoch 6 / 10  loss : 0.9304  acc : 0.4418
epoch 7 / 10  loss : 0.9315  acc : 0.4323
epoch 8 / 10  loss : 0.9096  acc : 0.4114
epoch 9 / 10  loss : 0.9691  acc : 0.4291
epoch 10 / 10  loss : 0.9119  acc : 0.3563


In [84]:
train_and_evaluation('GRU',10)

epoch 1 / 10  loss : 1.0469  acc : 0.4722
epoch 2 / 10  loss : 0.9602  acc : 0.5095
epoch 3 / 10  loss : 0.8804  acc : 0.5272
epoch 4 / 10  loss : 0.7731  acc : 0.5424
epoch 5 / 10  loss : 0.6468  acc : 0.5342
epoch 6 / 10  loss : 0.5034  acc : 0.5158
epoch 7 / 10  loss : 0.3615  acc : 0.5051
epoch 8 / 10  loss : 0.2654  acc : 0.5215
epoch 9 / 10  loss : 0.1993  acc : 0.5437
epoch 10 / 10  loss : 0.1571  acc : 0.5329


In [85]:
train_and_evaluation('LSTM',10)

epoch 1 / 10  loss : 1.0512  acc : 0.4810
epoch 2 / 10  loss : 0.9812  acc : 0.4994
epoch 3 / 10  loss : 0.8979  acc : 0.5297
epoch 4 / 10  loss : 0.7983  acc : 0.5253
epoch 5 / 10  loss : 0.6920  acc : 0.5513
epoch 6 / 10  loss : 0.5987  acc : 0.5430
epoch 7 / 10  loss : 0.5022  acc : 0.5475
epoch 8 / 10  loss : 0.4022  acc : 0.5323
epoch 9 / 10  loss : 0.3497  acc : 0.5437
epoch 10 / 10  loss : 0.2824  acc : 0.5456


In [83]:
train_and_evaluation('CNN',10)

epoch 1 / 10  loss : 1.0327  acc : 0.4810
epoch 2 / 10  loss : 0.7879  acc : 0.4962
epoch 3 / 10  loss : 0.5676  acc : 0.5133
epoch 4 / 10  loss : 0.3487  acc : 0.5095
epoch 5 / 10  loss : 0.1870  acc : 0.5152
epoch 6 / 10  loss : 0.0971  acc : 0.5152
epoch 7 / 10  loss : 0.0556  acc : 0.5228
epoch 8 / 10  loss : 0.0352  acc : 0.5209
epoch 9 / 10  loss : 0.0247  acc : 0.5146
epoch 10 / 10  loss : 0.0187  acc : 0.5152


# [8교시]